# Visualize OT alignment before and after training

Notebook cung cấp một API đơn giản để so sánh OT alignment của base model và adapter/checkpoint. Chạy các cell định nghĩa **một lần**, sau đó chỉ sửa cell **CHẠY Ở ĐÂY** cuối notebook để đổi câu, adapter và tên ảnh.

Hàm chính:

```python
result = visualize_ot_alignment(...)
```

`result` chứa `before`, `after`, `metrics`, `figure` và `saved_to`. Hai model được nạp tuần tự để tiết kiệm VRAM.


In [ ]:
from pathlib import Path
import gc
import sys
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from IPython.display import display
from peft import PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'src').exists(), 'Hãy mở notebook từ thư mục repo hoặc notebooks/.'

sys.path.insert(0, str(REPO_ROOT))
from src.prepare_data import prepare_alignment_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
         else torch.float16 if torch.cuda.is_available() else torch.float32)
sns.set_theme(style='white')
print(f'Device: {DEVICE} | dtype: {DTYPE}')


## Các hàm OT

Các tham số mặc định của solver khớp với Stage 1. Có thể override chúng ngay trong lời gọi hàm cuối notebook.


In [ ]:
def mixed_mass(scores, alpha):
    scores = scores.float().clamp_min(0)
    uniform = torch.full_like(scores, 1.0 / scores.numel())
    attention_mass = scores / scores.sum().clamp_min(1e-8)
    if scores.sum() <= 1e-8:
        attention_mass = uniform
    return alpha * attention_mass + (1.0 - alpha) * uniform


def sinkhorn_plan(cost, source_mass, target_mass, epsilon, iterations):
    log_a = source_mass.clamp_min(1e-38).log()
    log_b = target_mass.clamp_min(1e-38).log()
    log_k = -cost.float() / epsilon
    log_u = torch.zeros_like(source_mass, dtype=torch.float32)
    log_v = torch.zeros_like(target_mass, dtype=torch.float32)
    for _ in range(iterations):
        log_u = log_a - torch.logsumexp(log_k + log_v.unsqueeze(0), dim=1)
        log_v = log_b - torch.logsumexp(log_k.T + log_u.unsqueeze(0), dim=1)
    plan = torch.exp(log_u[:, None] + log_k + log_v[None, :])
    return plan / plan.sum().clamp_min(1e-8)


def ipot_plan(cost, source_mass, target_mass, beta, iterations, inner_iterations):
    log_a = source_mass.clamp_min(1e-38).log()
    log_b = target_mass.clamp_min(1e-38).log()
    log_kernel = -cost.float() / beta
    log_transport = log_a[:, None] + log_b[None, :]
    log_v = torch.zeros_like(target_mass, dtype=torch.float32)
    for _ in range(iterations):
        log_q = log_kernel + log_transport
        for _ in range(inner_iterations):
            log_u = log_a - torch.logsumexp(log_q + log_v.unsqueeze(0), dim=1)
            log_v = log_b - torch.logsumexp(log_q.T + log_u.unsqueeze(0), dim=1)
        log_transport = log_u[:, None] + log_q + log_v[None, :]
    plan = torch.exp(log_transport)
    return plan / plan.sum().clamp_min(1e-8)


def solve_plan(cost, source_mass, target_mass, config):
    if config['ot_solver'] == 'sinkhorn':
        return sinkhorn_plan(cost, source_mass, target_mass,
                             config['sinkhorn_epsilon'], config['sinkhorn_iterations'])
    if config['ot_solver'] == 'ipot':
        return ipot_plan(cost, source_mass, target_mass, config['ipot_beta'],
                         config['ipot_iterations'], config['ipot_inner_iterations'])
    raise ValueError("ot_solver phải là 'sinkhorn' hoặc 'ipot'")


## Chuẩn bị input và trích xuất alignment

Mọi dữ liệu thay đổi theo lần chạy được truyền qua `context`/`config`; các hàm không còn đọc câu, tokenizer hoặc span từ biến global.


In [ ]:
def _is_adapter(model_path):
    path = Path(str(model_path))
    return path.is_dir() and (path / 'adapter_config.json').exists()


def build_context(source_text, target_text, source_lang, target_lang,
                  tokenizer_source, prompt_format='plain', enable_thinking=False,
                  training_mode='finetune', trust_remote_code=False):
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source, trust_remote_code=trust_remote_code
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    raw = DatasetDict({'test': Dataset.from_list([{
        'source': source_text,
        'target': target_text,
        'source_lang': source_lang,
        'target_lang': target_lang,
    }])})
    sample = prepare_alignment_dataset(
        raw, tokenizer, prompt_format=prompt_format,
        enable_thinking=enable_thinking, training_mode=training_mode,
    )['test'][0]
    input_ids = torch.tensor(sample['input_ids'], dtype=torch.long).unsqueeze(0)
    ss, se = sample['source_start_positions'], sample['source_end_positions']
    ts, te = sample['target_start_positions'], sample['target_end_positions']
    assert 0 <= ss < se <= ts < te <= input_ids.shape[1]
    return {
        'tokenizer': tokenizer,
        'input_ids': input_ids,
        'attention_mask': torch.ones_like(input_ids),
        'spans': (ss, se, ts, te),
        'source_text': source_text,
        'target_text': target_text,
    }


def readable_tokens(token_ids, tokenizer):
    tokens = tokenizer.convert_ids_to_tokens(list(token_ids))
    return [f'{i}: ' + token.replace('Ġ', '␠').replace('▁', '␠').replace('\n', '↵')
            for i, token in enumerate(tokens)]


def load_causal_lm(model_path, adapter, base_model_name, config):
    kwargs = {
        'trust_remote_code': config['trust_remote_code'],
        'attn_implementation': 'eager',
        'torch_dtype': DTYPE,
    }
    if DEVICE == 'cuda':
        kwargs['device_map'] = 'auto'
    if adapter:
        base = AutoModelForCausalLM.from_pretrained(base_model_name, **kwargs)
        model = PeftModel.from_pretrained(base, model_path)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_path, **kwargs)
    if DEVICE == 'cpu':
        model = model.to(DEVICE)
    model.eval()
    return model


@torch.inference_mode()
def extract_alignment(model, context, config):
    tokenizer = context['tokenizer']
    input_ids = context['input_ids']
    attention_mask = context['attention_mask']
    ss, se, ts, te = context['spans']
    forward_mode = config['alignment_forward_mode']
    if forward_mode not in {'joint', 'independent'}:
        raise ValueError("alignment_forward_mode phải là 'joint' hoặc 'independent'")

    input_device = model.get_input_embeddings().weight.device
    align_layer = config['align_layer']
    attention_layer = align_layer if align_layer < 0 else max(align_layer - 1, 0)

    def forward(ids, mask):
        return model(
            input_ids=ids.to(input_device), attention_mask=mask.to(input_device),
            output_hidden_states=True, output_attentions=True,
            use_cache=False, return_dict=True,
        )

    if forward_mode == 'joint':
        outputs = forward(input_ids, attention_mask)
        hidden = outputs.hidden_states[align_layer][0].float()
        src_hidden, tgt_hidden = hidden[ss:se], hidden[ts:te]
        src_ids = input_ids[0, ss:se].tolist()
        tgt_ids = input_ids[0, ts:te].tolist()
        attention = outputs.attentions[attention_layer][0].float().mean(dim=0)
        received = attention[ts:te, :].sum(dim=0)
        source_scores, target_scores = received[ss:se], received[ts:te]
    else:
        batches = [tokenizer(
            text, add_special_tokens=config['independent_add_special_tokens'],
            return_tensors='pt', truncation=False,
        ) for text in (context['source_text'], context['target_text'])]
        outputs = [forward(batch['input_ids'], batch['attention_mask']) for batch in batches]
        special_ids = set(tokenizer.all_special_ids)
        all_ids = [batch['input_ids'][0].tolist() for batch in batches]
        indices = [[i for i, token_id in enumerate(ids) if token_id not in special_ids]
                   for ids in all_ids]
        if not indices[0] or not indices[1]:
            raise ValueError('Source hoặc target không có non-special token')
        selected_hidden, selected_scores = [], []
        for output, chosen in zip(outputs, indices):
            hidden = output.hidden_states[align_layer][0].float()
            index = torch.tensor(chosen, device=hidden.device)
            selected_hidden.append(hidden.index_select(0, index))
            attention = output.attentions[attention_layer][0].float().mean(dim=0)
            attn_index = torch.tensor(chosen, device=attention.device)
            selected = attention.index_select(0, attn_index).index_select(1, attn_index)
            selected_scores.append(selected.sum(dim=0))
        src_hidden, tgt_hidden = selected_hidden
        source_scores, target_scores = selected_scores
        src_ids = [all_ids[0][i] for i in indices[0]]
        tgt_ids = [all_ids[1][i] for i in indices[1]]

    similarity = F.normalize(src_hidden, dim=-1) @ F.normalize(tgt_hidden, dim=-1).T
    cost = 1.0 - similarity
    source_mass = mixed_mass(source_scores, config['attention_mass_weight']).to(cost.device)
    target_mass = mixed_mass(target_scores, config['attention_mass_weight']).to(cost.device)
    plan = solve_plan(cost, source_mass, target_mass, config)
    probability = plan / plan.sum().clamp_min(1e-8)
    entropy = -(probability * probability.clamp_min(1e-38).log()).sum()
    entropy /= np.log(max(probability.numel(), 2))
    src_pos = torch.linspace(0, 1, len(src_ids), device=plan.device)[:, None]
    tgt_pos = torch.linspace(0, 1, len(tgt_ids), device=plan.device)[None, :]
    band_mass = (plan * ((src_pos - tgt_pos).abs() <= 0.15)).sum()
    return {
        'similarity': similarity.cpu().numpy(),
        'plan': plan.cpu().numpy(),
        'source_mass': source_mass.cpu().numpy(),
        'target_mass': target_mass.cpu().numpy(),
        'source_tokens': readable_tokens(src_ids, tokenizer),
        'target_tokens': readable_tokens(tgt_ids, tokenizer),
        'source_token_ids': src_ids,
        'target_token_ids': tgt_ids,
        'metrics': {
            'OT expected cost ↓': float((plan * cost).sum()),
            'Mean best cosine ↑': float(similarity.max(dim=1).values.mean()),
            'Normalized plan entropy': float(entropy),
            'Monotonic-band mass ↑ (heuristic)': float(band_mass),
        },
    }


def analyze_checkpoint(model_path, context, config, name, adapter='auto'):
    adapter = _is_adapter(model_path) if adapter == 'auto' else bool(adapter)
    base_model_name = config['base_model_name']
    if adapter:
        base_model_name = PeftConfig.from_pretrained(str(model_path)).base_model_name_or_path
    print(f'Đang đo {name}: {model_path} | adapter={adapter}')
    model = load_causal_lm(str(model_path), adapter, base_model_name, config)
    try:
        result = extract_alignment(model, context, config)
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    result.update(name=name, model_path=str(model_path))
    return result


## Vẽ, lưu ảnh và hàm chạy chính

Mặc định heatmap transport được gộp subword thành word để dễ đọc. Truyền `merge_subwords=False` nếu muốn xem từng token gốc.


In [ ]:
def _word_groups(token_ids, tokenizer):
    token_strings = tokenizer.convert_ids_to_tokens(token_ids)
    groups = []
    special_ids = set(tokenizer.all_special_ids)
    for index, (token_id, token_string) in enumerate(zip(token_ids, token_strings)):
        piece = tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
        stripped = piece.strip()
        starts_word = token_string.startswith(('Ġ', '▁')) or piece[:1].isspace()
        is_continuation = token_string.startswith('##')
        has_cjk = any(('\u3400' <= char <= '\u9fff') or ('\uac00' <= char <= '\ud7af')
                      for char in stripped)
        is_punctuation = bool(stripped) and all(
            unicodedata.category(char)[0] in {'P', 'S'} for char in stripped
        )
        start_new = (not groups or token_id in special_ids or starts_word or is_punctuation
                     or (has_cjk and not is_continuation))
        if start_new:
            groups.append([index])
        else:
            groups[-1].append(index)
    labels = []
    for group in groups:
        ids = [token_ids[index] for index in group]
        label = tokenizer.decode(ids, clean_up_tokenization_spaces=False).strip()
        labels.append(label or ''.join(token_strings[index] for index in group))
    return groups, labels


def merge_subword_matrix(matrix, source_ids, target_ids, tokenizer, reducer='sum'):
    source_groups, source_labels = _word_groups(source_ids, tokenizer)
    target_groups, target_labels = _word_groups(target_ids, tokenizer)
    merged = np.zeros((len(source_groups), len(target_groups)), dtype=matrix.dtype)
    for row, source_group in enumerate(source_groups):
        for column, target_group in enumerate(target_groups):
            values = matrix[np.ix_(source_group, target_group)]
            merged[row, column] = values.sum() if reducer == 'sum' else values.mean()
    return merged, source_labels, target_labels


def _plot_data(result, tokenizer, merge_subwords):
    if not merge_subwords:
        return (result['similarity'], result['plan'],
                result['source_tokens'], result['target_tokens'])
    similarity, source_labels, target_labels = merge_subword_matrix(
        result['similarity'], result['source_token_ids'], result['target_token_ids'],
        tokenizer, reducer='mean',
    )
    plan, _, _ = merge_subword_matrix(
        result['plan'], result['source_token_ids'], result['target_token_ids'],
        tokenizer, reducer='sum',
    )
    return similarity, plan, source_labels, target_labels


def plot_alignment_comparison(before, after, tokenizer, output_path=None,
                              merge_subwords=True, show=True, dpi=180,
                              similarity_cmap='coolwarm', transport_cmap='YlOrRd'):
    before_sim, before_plan, before_src, before_tgt = _plot_data(
        before, tokenizer, merge_subwords
    )
    after_sim, after_plan, after_src, after_tgt = _plot_data(
        after, tokenizer, merge_subwords
    )
    sim_min = min(before_sim.min(), after_sim.min())
    sim_max = max(before_sim.max(), after_sim.max())
    plan_max = max(before_plan.max(), after_plan.max())
    width = max(15, 0.45 * max(len(before_tgt), len(after_tgt)) * 2)
    height = max(10, 0.35 * (len(before_src) + len(after_src)))
    fig, axes = plt.subplots(2, 2, figsize=(width, height), constrained_layout=True)

    plots = [
        (axes[0, 0], before_sim, 'BEFORE · cosine similarity', similarity_cmap, sim_min, sim_max,
         before_src, before_tgt),
        (axes[0, 1], after_sim, 'AFTER · cosine similarity', similarity_cmap, sim_min, sim_max,
         after_src, after_tgt),
        (axes[1, 0], before_plan, 'BEFORE · transport plan', transport_cmap, 0, plan_max,
         before_src, before_tgt),
        (axes[1, 1], after_plan, 'AFTER · transport plan', transport_cmap, 0, plan_max,
         after_src, after_tgt),
    ]
    for ax, matrix, title, cmap, vmin, vmax, source_labels, target_labels in plots:
        sns.heatmap(matrix, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
                    xticklabels=target_labels, yticklabels=source_labels)
        ax.set(title=title, xlabel='Target', ylabel='Source')
        ax.tick_params(axis='x', rotation=70, labelsize=8)
        ax.tick_params(axis='y', rotation=0, labelsize=8)

    saved_to = None
    if output_path is not None:
        saved_to = Path(output_path)
        if not saved_to.is_absolute():
            saved_to = REPO_ROOT / saved_to
        saved_to.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(saved_to, dpi=dpi, bbox_inches='tight')
        print(f'Đã lưu ảnh: {saved_to.resolve()}')
    if show:
        plt.show()
    return fig, saved_to


def visualize_ot_alignment(
    source_text,
    target_text,
    after_model_or_adapter,
    before_model_name_or_path=None,
    source_lang='en',
    target_lang='vi',
    output_path='outputs/ot-visualization/alignment.png',
    *,
    prompt_format='plain',
    enable_thinking=False,
    training_mode='finetune',
    alignment_forward_mode='joint',
    independent_add_special_tokens=True,
    align_layer=-1,
    ot_solver='sinkhorn',
    attention_mass_weight=0.5,
    sinkhorn_epsilon=0.1,
    sinkhorn_iterations=20,
    ipot_beta=0.5,
    ipot_iterations=50,
    ipot_inner_iterations=1,
    trust_remote_code=False,
    merge_subwords=True,
    similarity_cmap='coolwarm',
    transport_cmap='YlOrRd',
    show=True,
    dpi=180,
):
    """Run BEFORE/AFTER alignment, draw heatmaps, optionally save, and return artifacts."""
    after_is_adapter = _is_adapter(after_model_or_adapter)
    adapter_base = (PeftConfig.from_pretrained(str(after_model_or_adapter)).base_model_name_or_path
                    if after_is_adapter else None)
    base_model_name = before_model_name_or_path or adapter_base
    if base_model_name is None:
        raise ValueError('Hãy truyền before_model_name_or_path khi AFTER không phải PEFT adapter.')
    if adapter_base and before_model_name_or_path and before_model_name_or_path != adapter_base:
        print(f'Cảnh báo: adapter dùng base {adapter_base}, nhưng BEFORE là {before_model_name_or_path}.')

    after_path = Path(str(after_model_or_adapter))
    has_tokenizer = after_path.is_dir() and any(
        (after_path / name).exists() for name in ('tokenizer.json', 'tokenizer_config.json')
    )
    tokenizer_source = str(after_model_or_adapter) if has_tokenizer else base_model_name
    context = build_context(
        source_text, target_text, source_lang, target_lang, tokenizer_source,
        prompt_format, enable_thinking, training_mode, trust_remote_code,
    )
    config = {
        'base_model_name': base_model_name,
        'alignment_forward_mode': alignment_forward_mode,
        'independent_add_special_tokens': independent_add_special_tokens,
        'align_layer': align_layer,
        'ot_solver': ot_solver,
        'attention_mass_weight': attention_mass_weight,
        'sinkhorn_epsilon': sinkhorn_epsilon,
        'sinkhorn_iterations': sinkhorn_iterations,
        'ipot_beta': ipot_beta,
        'ipot_iterations': ipot_iterations,
        'ipot_inner_iterations': ipot_inner_iterations,
        'trust_remote_code': trust_remote_code,
    }
    ss, se, ts, te = context['spans']
    ids = context['input_ids'][0]
    print('Source span:', context['tokenizer'].decode(ids[ss:se]))
    print('Target span:', context['tokenizer'].decode(ids[ts:te]))

    before = analyze_checkpoint(base_model_name, context, config, 'BEFORE', adapter=False)
    after = analyze_checkpoint(after_model_or_adapter, context, config, 'AFTER', adapter='auto')
    metrics = pd.DataFrame({'BEFORE': before['metrics'], 'AFTER': after['metrics']})
    metrics['DELTA (AFTER - BEFORE)'] = metrics['AFTER'] - metrics['BEFORE']
    figure, saved_to = plot_alignment_comparison(
        before, after, context['tokenizer'], output_path,
        merge_subwords=merge_subwords, show=show, dpi=dpi,
        similarity_cmap=similarity_cmap, transport_cmap=transport_cmap,
    )
    display(metrics)
    return {
        'before': before,
        'after': after,
        'metrics': metrics,
        'figure': figure,
        'saved_to': saved_to,
        'context': context,
        'config': config,
    }


## CHẠY Ở ĐÂY — chỉ cần sửa cell bên dưới

- Đổi `SOURCE_TEXT`, `TARGET_TEXT` và `AFTER_MODEL_OR_ADAPTER`.
- `OUTPUT_PATH=None` nếu không muốn lưu; đường dẫn tương đối được tính từ thư mục repo.
- Đổi `SIMILARITY_CMAP` và `TRANSPORT_CMAP` để chọn màu heatmap (tên colormap của Matplotlib/Seaborn).
- Nếu AFTER là PEFT adapter, có thể để `BEFORE_MODEL=None`: notebook tự đọc base model từ `adapter_config.json`.
- Sau lần chạy đầu, chỉ cần chạy lại cell này cho câu/adapter mới; không cần chạy lại các cell định nghĩa.


In [ ]:
# ===== SỬA CẤU HÌNH Ở ĐÂY =====
SOURCE_TEXT = 'The World Health Organization reported 42 attacks on health facilities in May 2019.'
TARGET_TEXT = 'Tổ chức Y tế Thế giới báo cáo 42 vụ tấn công vào các cơ sở y tế vào tháng 5 năm 2019.'
SOURCE_LANG = 'en'
TARGET_LANG = 'vi'

AFTER_MODEL_OR_ADAPTER = REPO_ROOT / 'outputs' / 'stage1-multilingual-alignment'
BEFORE_MODEL = None  # Tự lấy base model nếu AFTER là PEFT adapter
OUTPUT_PATH = REPO_ROOT / 'outputs' / 'ot-visualization' / 'en-vi-alignment.png'
SIMILARITY_CMAP = 'coolwarm'  # Sáng; có thể thử 'RdYlBu_r', 'Spectral_r'
TRANSPORT_CMAP = 'YlOrRd'     # Nền sáng; có thể thử 'Oranges', 'YlGnBu'

result = visualize_ot_alignment(
    source_text=SOURCE_TEXT,
    target_text=TARGET_TEXT,
    source_lang=SOURCE_LANG,
    target_lang=TARGET_LANG,
    after_model_or_adapter=AFTER_MODEL_OR_ADAPTER,
    before_model_name_or_path=BEFORE_MODEL,
    output_path=OUTPUT_PATH,
    # Các tùy chọn thường dùng:
    alignment_forward_mode='joint',
    align_layer=-1,
    ot_solver='sinkhorn',
    merge_subwords=True,
    similarity_cmap=SIMILARITY_CMAP,
    transport_cmap=TRANSPORT_CMAP,
    show=True,
)
